In [ ]:
from utils.generic_utils import load_all_games_csv, basic_win_prob_for_et, predict_lr
from utils.elo_tracker_utils import add_elos_to_games_df, evaluate_elo_prob_func
from scipy.special import expit
import numpy as np

# Win Probability Analysis

This notebook will compare several different methods to estimate win probabilities from Elo ratings, and possibly home advantage, travel distance, and rest days.

## Get all Games

In [2]:
all_games = load_all_games_csv('../data/gameinfo_cleaned.csv')
#all_games = all_games[(all_games['season'] >= 1990) & (all_games['season'] < 2000)]
all_games.head()

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/utils.py:27: DtypeWarning: Columns (10,11,13,17,19,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(filename)


,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs,homelastkwinpct,vislastkwinpct
gid,,,,,,,,,,,,,,,,,,,,,
LS3189904140,CHN,LS3,LOU03,18990414,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
PHI189904140,WSN,PHI,PHI09,18990414,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BLN189904150,NY1,BLN,BAL07,18990415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BRO189904150,BSN,BRO,NYC12,18990415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
CIN189904150,PIT,CIN,CIN05,18990415,0.0,0:00PM,day,NaN,NaN,False,...,NaN,NaN,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN


In [3]:
# Max rest days
all_games['visrestdays'] = all_games['visrestdays'].apply(lambda x: min(3,x))
all_games['homerestdays'] = all_games['homerestdays'].apply(lambda x: min(3,x))


In [4]:
# Take cube root of distance traveled
all_games['homedistancetraveled'] = all_games['homedistancetraveled']**(1/3)
all_games['visdistancetraveled'] = all_games['visdistancetraveled']**(1/3)

In [5]:
# Take square root of margin of victory
all_games['marginofvictory'] = np.sqrt(all_games['marginofvictory'])
all_games.head()

,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,homerestdays,visrestdays,homepitcherrgs,vispitcherrgs,hometeamrgs,visteamrgs,homepitcherminusteamrgs,vispitcherminusteamrgs,homelastkwinpct,vislastkwinpct
gid,,,,,,,,,,,,,,,,,,,,,
LS3189904140,CHN,LS3,LOU03,18990414,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
PHI189904140,WSN,PHI,PHI09,18990414,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BLN189904150,NY1,BLN,BAL07,18990415,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
BRO189904150,BSN,BRO,NYC12,18990415,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN
CIN189904150,PIT,CIN,CIN05,18990415,0.0,0:00PM,day,NaN,NaN,False,...,3.0,3.0,39.0,39.0,39.0,39.0,0.0,0.0,NaN,NaN


In [6]:
# Subtract 0.5 to last k % to go between +/- 0.5 - any na becomes 0
all_games['homemomentum'] = all_games['homelastkwinpct']#.apply(lambda x: 0 if np.isnan(x) else x - 0.5)
all_games['vismomentum'] = all_games['vislastkwinpct']#.apply(lambda x: 0 if np.isnan(x) else x - 0.5)

## Evaluate Simple Probability model

In [39]:
bce, accuracy = evaluate_elo_prob_func(all_games, basic_win_prob_for_et, K=3, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6806169693165345
Accuracy: 0.5630001342809257


## With +28 Adjustment for Home Team

In [40]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo + 28, away_elo, game_info), K=2, use_margin_of_victory=True, skip_first_n=20)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6775131161851833
Accuracy: 0.569850811847177


## With + 1.9% Adjustment for Home Team

In [41]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo*1.019, away_elo, game_info), K=2, use_margin_of_victory=True, skip_first_n=20)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6775096408932033
Accuracy: 0.5695362585129117


## With Logistic Regression

In [9]:
# First need to fit using some Elos - use the initial basic probability func.

df_to_fit = add_elos_to_games_df(all_games, use_margin_of_victory=True, K=3)


In [96]:
df_to_fit['elodiff'] = df_to_fit['viselo'] - df_to_fit['homeelo']
df_to_fit['restdiff'] = df_to_fit['visrestdays'] - df_to_fit['homerestdays']
df_to_fit['distancediff'] = df_to_fit['visdistancetraveled'] - df_to_fit['homedistancetraveled']
df_to_fit['homediff'] =  0 - 1
df_to_fit['pitcherdiff'] = df_to_fit['vispitcherminusteamrgs'] - df_to_fit['homepitcherminusteamrgs']
df_to_fit['momentumdiff'] = df_to_fit['vismomentum'] - df_to_fit['homemomentum']

#features = ['elodiff', 'distancediff', 'restdiff']
features = ['elodiff', 'homediff', 'restdiff', 'distancediff', 'pitcherdiff', 'momentumdiff']
X = df_to_fit[features].to_numpy()
y = df_to_fit['homewon'].astype(int).to_numpy().reshape(-1,1)

s = -np.log(10) / 400

In [97]:
# Fit via GD
w = np.zeros((6,1))
w[0,0] = s # Becomes 1 once dividing by s

step = 0.01 # Slightly higher for small gradients
iterations = 10000

for _ in range(iterations):

    z = X @ w
    y_hat = expit(z)
    
    w_grad = (1/X.shape[0]) * X.T @ (y_hat - y)
    
    w_grad[0,0] = 0
    
    #print(w_grad)
    
    w = w - step*w_grad
    
w

array([[-0.00575646],
       [-0.1568339 ],
       [-0.02507092],
       [ 0.00177641],
       [-0.00897878],
       [ 0.04983321]])

In [98]:
# Convert back to interpretable coefficients for individual Elo adjustments
w = (1/s) * w
w

array([[ 1.        ],
       [27.2448388 ],
       [ 4.35526567],
       [-0.30859427],
       [ 1.55977433],
       [-8.65691528]])

In [ ]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w), K=3, use_margin_of_victory=True, skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6764713877645896
Accuracy: 0.571897225282981
